<a href="https://colab.research.google.com/github/revadirevathi20-sys/hdb-resale-etl-pipeline/blob/main/notebooks/hdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


HDB Resale ETL Pipeline - Full Notebook

This notebook runs the full ETL pipeline and displays all required outputs for verification against the assignment requirements

Requirements covered:
Data Quality  : Combine, Profile, Validate, Remaining Lease, Dedup, Anomalies
Transformation: Resale Identifier, Dedup, Hashing
Output        : Raw, Cleaned, Transformed, Failed, Hashed


In [23]:
# import shutil
# import os

# paths_to_delete = [
#     "/content/hdb-resale-etl-pipeline",
#     "/content/data"
# ]

# for path in paths_to_delete:
#     if os.path.exists(path):
#         shutil.rmtree(path)
#         print(f"✓ Deleted {path}")
#     else:
#         print(f"✗ Not found: {path}")

✓ Deleted /content/hdb-resale-etl-pipeline
✓ Deleted /content/data


In [24]:
# import os

# # Reset to /content first since the previous working directory was deleted
# os.chdir("/content")

# repo_name = "hdb-resale-etl-pipeline"

# if not os.path.exists(f"/content/{repo_name}"):
#     !git clone https://github.com/revadirevathi20-sys/hdb-resale-etl-pipeline.git

# os.chdir(f"/content/{repo_name}")
# print("✓ Working directory:", os.getcwd())

Cloning into 'hdb-resale-etl-pipeline'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 299 (delta 77), reused 15 (delta 15), pack-reused 159 (from 1)
Receiving objects: 100% (299/299), 10.93 MiB | 17.59 MiB/s, done.
Resolving deltas: 100% (147/147), done.
✓ Working directory: /content/hdb-resale-etl-pipeline


In [25]:
# Clone the repository and set working directory

import os

repo_name = "hdb-resale-etl-pipeline"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/revadirevathi20-sys/hdb-resale-etl-pipeline.git

os.chdir(f"/content/{repo_name}")
print("✓ Working directory:", os.getcwd())

✓ Working directory: /content/hdb-resale-etl-pipeline


In [26]:
# Import Dependencies

import pandas as pd
import numpy as np
import hashlib
import warnings
warnings.filterwarnings("ignore")

print("✓ Dependencies loaded")

✓ Dependencies loaded


In [27]:
# Run Ingestion (Data Quality Req 1 — Combine Datasets)
# Combines all raw CSV files into a single master dataset filtered to Jan 2012 to Dec 2016
# Saves to data/cleaned/master_raw_combined.csv.

import subprocess, sys

print("=" * 60)
print("STEP 1: INGESTION — Combining raw datasets")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "src/ingest.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

STEP 1: INGESTION — Combining raw datasets
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv
True
[ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv] Filtered out 22281 rows before 2012-01
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv
True
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv
True
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv
True

Master dataset shape: (92544, 11)
Date range: 2012-01-01 00:00:00 to 2016-12-01 00:00:00
Master dataset saved to ../data/combined/master_raw_combined.csv



In [28]:
# Verify Raw Output
# Data Output Req — RAW: Displays the combined master dataset as-is.

print("=" * 60)
print("OUTPUT — RAW: master_raw_combined.csv")
print("=" * 60)

raw = pd.read_csv("../data/combined/master_raw_combined.csv", low_memory=False)

print(f"Shape       : {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Date range  : {raw['month'].min()} → {raw['month'].max()}")
print(f"Columns     : {list(raw.columns)}")
print()
print("── First 5 rows ──")
display(raw.head())

print()
print("── Last 5 rows ──")
display(raw.tail())

OUTPUT — RAW: master_raw_combined.csv
Shape       : 92,544 rows × 11 columns
Date range  : 2012-01-01 → 2016-12-01
Columns     : ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']

── First 5 rows ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2012-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,NaN
1,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,NaN
2,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,NaN
3,2012-01-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,NaN
4,2012-01-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,NaN



── Last 5 rows ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
92539,2016-12-01,YISHUN,5 ROOM,297,YISHUN ST 20,13 TO 15,112.0,Improved,2000,488000.0,82.0
92540,2016-12-01,YISHUN,5 ROOM,838,YISHUN ST 81,01 TO 03,122.0,Improved,1987,455000.0,69.0
92541,2016-12-01,YISHUN,EXECUTIVE,664,YISHUN AVE 4,10 TO 12,181.0,Apartment,1992,778000.0,74.0
92542,2016-12-01,YISHUN,EXECUTIVE,325,YISHUN CTRL,01 TO 03,146.0,Maisonette,1988,575000.0,70.0
92543,2016-12-01,YISHUN,MULTI-GENERATION,666,YISHUN AVE 4,10 TO 12,164.0,Multi Generation,1987,735000.0,70.0


In [29]:
# Run Cleaning (Data Quality Req 2–7)
# Performs:
#   Req 2 — Data profiling
#   Req 3 — Validation rules (date, town, flat type, flat model, storey_range)
#   Req 4 — Recompute remaining lease (99-year HDB lease, rounded down)
#   Req 5 — Duplicate key handling (keep higher resale price)
#   Req 6 — Anomalous price detection (z-score heuristic)
#   Req 7 — Additional cleaning (whitespace, casing, floor area bounds)

print("=" * 60)
print("STEP 2: CLEANING — Data Quality Requirements 2–7")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "src/clean.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

STEP 2: CLEANING — Data Quality Requirements 2–7

========== START CLEANING ==========

=== DATA PROFILING ===
Shape: (92544, 11)

Null counts:
month                      0
town                       0
flat_type                  0
block                      0
street_name                0
storey_range               0
floor_area_sqm             0
flat_model                 0
lease_commence_date        0
resale_price               0
remaining_lease        55391
dtype: int64

Dtypes:
month                  datetime64[ns]
town                           object
flat_type                      object
block                          object
street_name                    object
storey_range                   object
floor_area_sqm                float64
flat_model                     object
lease_commence_date             int64
resale_price                  float64
remaining_lease               float64
dtype: object

Duplicates: 273

Numeric summary:
                               month  ...  remai

In [30]:
# Verify Cleaned Output
# Data Output Req — CLEANED: Dataset that passes all data quality requirements.

print("=" * 60)
print("OUTPUT — CLEANED: hdb_resale_cleaned.csv")
print("=" * 60)

cleaned = pd.read_csv("../data/cleaned/hdb_resale_cleaned.csv", low_memory=False)

print(f"Shape             : {cleaned.shape[0]:,} rows × {cleaned.shape[1]} columns")
print(f"Null counts       :\n{cleaned.isnull().sum()}")
print()
print("── Sample rows ──")
display(cleaned.head(10))

print()
print("── Remaining Lease column (Data Quality Req 4) ──")
display(cleaned[["month", "lease_commence_date", "remaining_lease", "lease_end_year"]].head(10))

print()
print("── Unique towns in cleaned data ──")
print(sorted(cleaned["town"].unique()))

print()
print("── Unique flat types in cleaned data ──")
print(sorted(cleaned["flat_type"].unique()))

print()
print("── Resale price statistics ──")
display(cleaned["resale_price"].describe())

OUTPUT — CLEANED: hdb_resale_cleaned.csv
Shape             : 90,475 rows × 12 columns
Null counts       :
month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
resale_price           0
remaining_lease        0
lease_end_year         0
dtype: int64

── Sample rows ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year
0,2016-09-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.0,TYPE S2,2011,1120000.0,93 YEARS 4 MONTHS,2110
1,2016-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,46 TO 48,107.0,TYPE S2,2011,1100000.0,93 YEARS 2 MONTHS,2110
2,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,TYPE S2,2011,1100000.0,93 YEARS 2 MONTHS,2110
3,2015-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,43 TO 45,107.0,TYPE S2,2011,1088000.0,94 YEARS 2 MONTHS,2110
4,2016-06-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,34 TO 36,107.0,TYPE S2,2011,1070000.0,93 YEARS 7 MONTHS,2110
5,2016-08-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,105.0,TYPE S2,2011,1070000.0,93 YEARS 5 MONTHS,2110
6,2016-01-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,28 TO 30,108.0,TYPE S2,2011,1068888.0,94 YEARS 0 MONTHS,2110
7,2016-05-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,19 TO 21,106.0,TYPE S2,2011,1063888.0,93 YEARS 8 MONTHS,2110
8,2015-04-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,28 TO 30,107.0,TYPE S2,2011,1060000.0,94 YEARS 9 MONTHS,2110
9,2016-09-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,40 TO 42,106.0,TYPE S2,2011,1055000.0,93 YEARS 4 MONTHS,2110



── Remaining Lease column (Data Quality Req 4) ──


,month,lease_commence_date,remaining_lease,lease_end_year
0,2016-09-01,2011,93 YEARS 4 MONTHS,2110
1,2016-11-01,2011,93 YEARS 2 MONTHS,2110
2,2016-11-01,2011,93 YEARS 2 MONTHS,2110
3,2015-11-01,2011,94 YEARS 2 MONTHS,2110
4,2016-06-01,2011,93 YEARS 7 MONTHS,2110
5,2016-08-01,2011,93 YEARS 5 MONTHS,2110
6,2016-01-01,2011,94 YEARS 0 MONTHS,2110
7,2016-05-01,2011,93 YEARS 8 MONTHS,2110
8,2015-04-01,2011,94 YEARS 9 MONTHS,2110
9,2016-09-01,2011,93 YEARS 4 MONTHS,2110



── Unique towns in cleaned data ──
['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']

── Unique flat types in cleaned data ──
['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']

── Resale price statistics ──


,resale_price
count,9.047500e+04
mean,4.507719e+05
std,1.273563e+05
min,1.920000e+05
25%,3.580000e+05
50%,4.280000e+05
75%,5.150000e+05
max,1.120000e+06


In [31]:
# Verify Failed Outputs
# Data Output Req — FAILED: Records removed at each cleaning stage.

print("=" * 60)
print("OUTPUT — FAILED: All failed record groups")
print("=" * 60)

failed_files = {
    "Validation failures"          : "../data/failed/failed_validation.csv",
    "Duplicate key failures"       : "../data/failed/failed_duplicates.csv",
    "Anomalous price failures"     : "../data/failed/failed_anomalies.csv",
    "Additional cleaning failures" : "../data/failed/failed_additional_cleaning.csv",
}

for label, path in failed_files.items():
    if os.path.exists(path):
        df_fail = pd.read_csv(path, low_memory=False)
        print(f"\n── {label}: {df_fail.shape[0]:,} records ──")
        display(df_fail.head(5))
    else:
        print(f"\n── {label}: file not found at {path} ──")

OUTPUT — FAILED: All failed record groups

── Validation failures: 0 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,fail_reason



── Duplicate key failures: 1,600 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,fail_reason
0,2013-07-01,BUKIT TIMAH,EXECUTIVE,7,TOH YI DR,04 TO 06,146.0,Maisonette,1989,950000.0,74 Years 6 Months,2088,duplicate_key_lower_price
1,2015-11-01,CENTRAL AREA,4 ROOM,1A,CANTONMENT RD,37 TO 39,95.0,Type S1,2011,915000.0,94 Years 2 Months,2110,duplicate_key_lower_price
2,2013-07-01,TOA PAYOH,5 ROOM,81,LOR 4 TOA PAYOH,19 TO 21,122.0,Improved,1997,880000.0,82 Years 6 Months,2096,duplicate_key_lower_price
3,2016-10-01,BISHAN,5 ROOM,273B,BISHAN ST 24,01 TO 03,120.0,DBSS,2011,878888.0,93 Years 3 Months,2110,duplicate_key_lower_price
4,2013-07-01,BISHAN,EXECUTIVE,187,BISHAN ST 13,07 TO 09,146.0,Maisonette,1987,878000.0,72 Years 6 Months,2086,duplicate_key_lower_price



── Anomalous price failures: 469 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,fail_reason
0,2016-12-01,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,1150000.0,54 Years 1 Months,2071,anomalous_price
1,2016-08-01,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.0,DBSS,2011,1100000.0,93 Years 5 Months,2110,anomalous_price
2,2014-10-01,BISHAN,EXECUTIVE,194,BISHAN ST 13,22 TO 24,150.0,Maisonette,1987,1088888.0,71 Years 3 Months,2086,anomalous_price
3,2015-03-01,KALLANG/WHAMPOA,3 ROOM,53,JLN MA'MOR,01 TO 03,280.0,Terrace,1972,1060000.0,55 Years 10 Months,2071,anomalous_price
4,2016-10-01,BISHAN,5 ROOM,273B,BISHAN ST 24,40 TO 42,120.0,DBSS,2011,1038000.0,93 Years 3 Months,2110,anomalous_price



── Additional cleaning failures: 0 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,fail_reason


In [34]:
# Cleaning Summary
# Quick reconciliation to confirm record counts add up correctly.

print("=" * 60)
print("CLEANING RECONCILIATION SUMMARY")
print("=" * 60)

raw_count       = pd.read_csv("../data/combined/master_raw_combined.csv").shape[0]
cleaned_count   = cleaned.shape[0]

failed_counts = {}
for label, path in failed_files.items():
    if os.path.exists(path):
        failed_counts[label] = pd.read_csv(path, low_memory=False).shape[0]
    else:
        failed_counts[label] = 0

total_failed = sum(failed_counts.values())

print(f"  Raw records              : {raw_count:>8,}")
for label, count in failed_counts.items():
    print(f"  {label:<30}: {count:>8,}")
print(f"  {'─'*42}")
print(f"  Final cleaned records    : {cleaned_count:>8,}")
print(f"  Check (raw - failed)     : {raw_count - total_failed:>8,}  ← should match cleaned")

CLEANING RECONCILIATION SUMMARY
  Raw records              :   92,544
  Validation failures           :        0
  Duplicate key failures        :    1,600
  Anomalous price failures      :      469
  Additional cleaning failures  :        0
  ──────────────────────────────────────────
  Final cleaned records    :   90,475
  Check (raw - failed)     :   90,475  ← should match cleaned


In [35]:
# Run Transformation (Transformation Req 1–3)
#   Performs:
#   Req 1 — Create Resale Identifier (S + block digits + avg price digits +
#            month digits + town initial)
#   Req 2 — Deduplicate on identifier (keep higher resale price)
#   Req 3 — Hash identifier using SHA-256 (irreversible, collision-resistant)

print("=" * 60)
print("STEP 3: TRANSFORMATION — Transformation Requirements 1–3")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "src/transform.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

STEP 3: TRANSFORMATION — Transformation Requirements 1–3

========== START TRANSFORMATION ==========

Transform duplicates: 13606 removed, 76869 kept

========== TRANSFORMATION SUMMARY ==========
Input records               : 90,475
Duplicate identifiers       : 13,606
Final transformed records   : 76,869
Final hashed records        : 76,869

Files saved:
../data/transformed/hdb_resale_transformed.csv
../data/hashed/hdb_resale_hashed.csv
../data/failed/failed_transform_duplicates.csv



In [37]:
# Verify Transformed Output
# Data Output Req — TRANSFORMED: Cleaned data with Resale Identifier added.
# Plain resale_identifier is retained here before hashing.

print("=" * 60)
print("OUTPUT — TRANSFORMED: hdb_resale_transformed.csv")
print("=" * 60)

transformed = pd.read_csv("../data/transformed/hdb_resale_transformed.csv", low_memory=False)

print(f"Shape   : {transformed.shape[0]:,} rows × {transformed.shape[1]} columns")
print(f"Columns : {list(transformed.columns)}")
print()
print("── Sample rows with Resale Identifier ──")
display(transformed[["month", "town", "flat_type", "block", "resale_price", "resale_identifier"]].head(10))

print()
print("── Identifier format check (should start with 'S') ──")
starts_with_s = transformed["resale_identifier"].str.startswith("S").all()
print(f"  All identifiers start with 'S': {starts_with_s}")

print()
print("── Identifier length distribution ──")
print(transformed["resale_identifier"].str.len().value_counts())

print()
print("── Uniqueness check ──")
total       = transformed.shape[0]
unique_ids  = transformed["resale_identifier"].nunique()
print(f"  Total records      : {total:,}")
print(f"  Unique identifiers : {unique_ids:,}")
print(f"  All unique         : {total == unique_ids}")

OUTPUT — TRANSFORMED: hdb_resale_transformed.csv
Shape   : 76,869 rows × 13 columns
Columns : ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease', 'lease_end_year', 'resale_identifier']

── Sample rows with Resale Identifier ──


,month,town,flat_type,block,resale_price,resale_identifier
0,2012-01-01,ANG MO KIO,4 ROOM,102,500000.0,S1024701A
1,2012-01-01,ANG MO KIO,3 ROOM,118,320000.0,S1183401A
2,2012-01-01,ANG MO KIO,3 ROOM,121,382800.0,S1213401A
3,2012-01-01,ANG MO KIO,4 ROOM,126,452000.0,S1264701A
4,2012-01-01,ANG MO KIO,3 ROOM,151,302000.0,S1513401A
5,2012-01-01,ANG MO KIO,3 ROOM,154,321000.0,S1543401A
6,2012-01-01,ANG MO KIO,3 ROOM,157,297000.0,S1573401A
7,2012-01-01,ANG MO KIO,3 ROOM,163,340000.0,S1633401A
8,2012-01-01,ANG MO KIO,2 ROOM,170,260000.0,S1702501A
9,2012-01-01,ANG MO KIO,3 ROOM,174,281000.0,S1743401A



── Identifier format check (should start with 'S') ──
  All identifiers start with 'S': True

── Identifier length distribution ──
resale_identifier
9    76869
Name: count, dtype: int64

── Uniqueness check ──
  Total records      : 76,869
  Unique identifiers : 76,869
  All unique         : True


In [38]:
# Verify Transform Failed Output
# Records removed during transformation deduplication (Transformation Req 2).

print("=" * 60)
print("OUTPUT — FAILED (Transform): failed_transform_duplicates.csv")
print("=" * 60)

transform_failed_path = "../data/failed/failed_transform_duplicates.csv"

if os.path.exists(transform_failed_path):
    tf_failed = pd.read_csv(transform_failed_path, low_memory=False)
    print(f"Shape   : {tf_failed.shape[0]:,} rows × {tf_failed.shape[1]} columns")
    print()
    display(tf_failed.head(10))
else:
    print("No transform duplicates found — file not created.")

OUTPUT — FAILED (Transform): failed_transform_duplicates.csv
Shape   : 13,606 rows × 14 columns



,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,resale_identifier,fail_reason
0,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,TYPE S2,2011,1100000.0,93 YEARS 2 MONTHS,2110,S0011011C,transform_duplicate_identifier
1,2016-09-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,40 TO 42,106.0,TYPE S2,2011,1055000.0,93 YEARS 4 MONTHS,2110,S0011009C,transform_duplicate_identifier
2,2015-04-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,46 TO 48,107.0,TYPE S2,2011,1050000.0,94 YEARS 9 MONTHS,2110,S0011004C,transform_duplicate_identifier
3,2015-02-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,37 TO 39,107.0,TYPE S2,2011,1030000.0,94 YEARS 11 MONTHS,2110,S0019902C,transform_duplicate_identifier
4,2016-01-01,CENTRAL AREA,5 ROOM,1F,CANTONMENT RD,28 TO 30,107.0,TYPE S2,2011,1020000.0,94 YEARS 0 MONTHS,2110,S0019401C,transform_duplicate_identifier
5,2015-02-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,22 TO 24,105.0,TYPE S2,2011,1000000.0,94 YEARS 11 MONTHS,2110,S0019902C,transform_duplicate_identifier
6,2016-09-01,CENTRAL AREA,5 ROOM,1F,CANTONMENT RD,22 TO 24,105.0,TYPE S2,2011,998000.0,93 YEARS 4 MONTHS,2110,S0011009C,transform_duplicate_identifier
7,2016-09-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,31 TO 33,106.0,TYPE S2,2011,995000.0,93 YEARS 4 MONTHS,2110,S0011009C,transform_duplicate_identifier
8,2015-02-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,28 TO 30,107.0,TYPE S2,2011,991000.0,94 YEARS 11 MONTHS,2110,S0019902C,transform_duplicate_identifier
9,2015-05-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,31 TO 33,107.0,TYPE S2,2011,980000.0,94 YEARS 8 MONTHS,2110,S0019505C,transform_duplicate_identifier


In [39]:
# Verify Hashed Output
# Data Output Req — HASHED: Cleaned data + hashed identifier column.
# Plain resale_identifier is dropped; only the SHA-256 hash is retained.
#
# Hashing algorithm: SHA-256
#   - Irreversible: cannot reconstruct original identifier from hash
#   - Deterministic: same input always produces same hash
#   - Collision-resistant: negligible probability of two different identifiers
#     producing the same hash (2^256 possible outputs)
#   - Uniqueness preserved: since identifiers are unique, their hashes are too

print("=" * 60)
print("OUTPUT — HASHED: hdb_resale_hashed.csv")
print("=" * 60)

hashed_path = "../data/hashed/hdb_resale_hashed.csv"

if os.path.exists(hashed_path):
    hashed = pd.read_csv(hashed_path, low_memory=False)

    print(f"Shape   : {hashed.shape[0]:,} rows × {hashed.shape[1]} columns")
    print(f"Columns : {list(hashed.columns)}")
    print()
    print("── Sample rows with hashed identifier ──")
    display(hashed[["month", "town", "flat_type", "resale_price", "resale_identifier_hashed"]].head(10))

    print()
    print("── Confirm plain resale_identifier is NOT present ──")
    plain_present = "resale_identifier" in hashed.columns
    print(f"  Plain identifier present: {plain_present}  ← should be False")

    print()
    print("── Hash format check (SHA-256 = 64 hex characters) ──")
    hash_lengths = hashed["resale_identifier_hashed"].str.len().value_counts()
    print(hash_lengths)
    all_64 = (hashed["resale_identifier_hashed"].str.len() == 64).all()
    print(f"  All hashes are 64 characters: {all_64}")

    print()
    print("── Hash uniqueness check ──")
    total_hashed  = hashed.shape[0]
    unique_hashes = hashed["resale_identifier_hashed"].nunique()
    print(f"  Total records      : {total_hashed:,}")
    print(f"  Unique hashes      : {unique_hashes:,}")
    print(f"  All unique         : {total_hashed == unique_hashes}")
else:
    print(f"Hashed file not found at {hashed_path}")

OUTPUT — HASHED: hdb_resale_hashed.csv
Shape   : 76,869 rows × 13 columns
Columns : ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease', 'lease_end_year', 'resale_identifier_hashed']

── Sample rows with hashed identifier ──


,month,town,flat_type,resale_price,resale_identifier_hashed
0,2012-01-01,ANG MO KIO,4 ROOM,500000.0,7bcc9678dc2e709618f1b02a0364a522b3fe9c10f5c9ef...
1,2012-01-01,ANG MO KIO,3 ROOM,320000.0,3e7d1629955bf8874d7b98e4bbd2aca86efb68c548d004...
2,2012-01-01,ANG MO KIO,3 ROOM,382800.0,ba7731ca106ec514d5471dfa92bd053d5c53b02d097b6b...
3,2012-01-01,ANG MO KIO,4 ROOM,452000.0,02e68094e7e2df51ad56ffac2c6f3f0adfd18236661dbf...
4,2012-01-01,ANG MO KIO,3 ROOM,302000.0,10882c6d534f79a06a8cbac9c62de905386e109232634c...
5,2012-01-01,ANG MO KIO,3 ROOM,321000.0,8dd77c00a6f9e876c0fc5fb1047a31c1d5de2a6e2bb3ac...
6,2012-01-01,ANG MO KIO,3 ROOM,297000.0,49cc5074d9830fd43968b4fc5bd073c1375c3501ab4ad4...
7,2012-01-01,ANG MO KIO,3 ROOM,340000.0,431b5100b470f01d7a355f4d65e7d522e22ee471ba3bc4...
8,2012-01-01,ANG MO KIO,2 ROOM,260000.0,7b76c2e5ad415d73d9ac4e74b364f51dc0b60f0db59b52...
9,2012-01-01,ANG MO KIO,3 ROOM,281000.0,6a21c3745f3b7a7ea6ef6486fb8f5a83dd68dc8e218df1...



── Confirm plain resale_identifier is NOT present ──
  Plain identifier present: False  ← should be False

── Hash format check (SHA-256 = 64 hex characters) ──
resale_identifier_hashed
64    76869
Name: count, dtype: int64
  All hashes are 64 characters: True

── Hash uniqueness check ──
  Total records      : 76,869
  Unique hashes      : 76,869
  All unique         : True


In [40]:
# Full Pipeline Summary
# End-to-end record count reconciliation across all output files.

print("=" * 60)
print("FULL PIPELINE SUMMARY")
print("=" * 60)

summary = {
    "1. Raw (combined)"                  : "../data/combined/master_raw_combined.csv",
    "2. Cleaned"                         : "../data/cleaned/hdb_resale_cleaned.csv",
    "3. Transformed"                     : "../data/transformed/hdb_resale_transformed.csv",
    "4. Hashed"                          : "../data/hashed/hdb_resale_hashed.csv",
    "5. Failed — Validation"             : "../data/failed/failed_validation.csv",
    "6. Failed — Duplicates (clean)"     : "../data/failed/failed_duplicates.csv",
    "7. Failed — Anomalous prices"       : "../data/failed/failed_anomalies.csv",
    "8. Failed — Additional cleaning"    : "../data/failed/failed_additional_cleaning.csv",
    "9. Failed — Transform duplicates"   : "../data/failed/failed_transform_duplicates.csv",
}

for label, path in summary.items():
    if os.path.exists(path):
        count = pd.read_csv(path, low_memory=False).shape[0]
        print(f"  {label:<40}: {count:>8,} records")
    else:
        print(f"  {label:<40}: {'file not found':>15}")

print()
print("All output files verified. Pipeline complete.")

FULL PIPELINE SUMMARY
  1. Raw (combined)                       :   92,544 records
  2. Cleaned                              :   90,475 records
  3. Transformed                          :   76,869 records
  4. Hashed                               :   76,869 records
  5. Failed — Validation                  :        0 records
  6. Failed — Duplicates (clean)          :    1,600 records
  7. Failed — Anomalous prices            :      469 records
  8. Failed — Additional cleaning         :        0 records
  9. Failed — Transform duplicates        :   13,606 records

All output files verified. Pipeline complete.
